# Basket Calibration — Learn Basket Location

**Purpose:** Calibrate the basket detector by positioning the robot near the basket and capturing its appearance.

**Instructions:**
1. Manually position the robot ~30 cm from the basket
2. Point the camera at the basket
3. Run all cells below
4. The calibration will be saved to `config.yaml`

**What this does:**
- Captures 10 frames of the basket
- Detects gray regions
- Calculates average size and location
- Saves calibration parameters

## 1. Setup

In [ ]:
import sys
import os
import time
import yaml
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets.widgets as widgets

# Add project root to path
project_root = '/home/steve/Notebooks/Projects/AI & Robotics Hackathon Berlin/ITQ_HackLab_Team_2'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from perception.basket_detector import BasketDetector

print("Setup complete!")

## 2. Initialize Camera

In [ ]:
# Try jetbot camera first, fallback to OpenCV
camera = None
camera_source = None

try:
    from jetbot import Camera
    camera = Camera.instance(width=320, height=240)
    camera_source = 'jetbot'
    print("✓ JetBot Camera initialized")
except Exception as e:
    print(f"JetBot camera failed: {e}")
    print("Trying OpenCV fallback...")
    
    try:
        camera = cv2.VideoCapture(0)
        camera.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
        camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)
        camera_source = 'opencv'
        print("✓ OpenCV Camera initialized")
    except Exception as e2:
        print(f"OpenCV camera failed: {e2}")
        print("ERROR: No camera available!")

if camera is None:
    print("\nTROUBLESHOOTING:")
    print("1. Check camera cable connection")
    print("2. Run: sudo pkill -f gst-launch")
    print("3. Kernel → Shutdown All Kernels → reopen notebook")

## 3. Test Camera Feed

In [ ]:
# Capture a test frame
def get_frame():
    if camera_source == 'jetbot':
        return camera.value
    elif camera_source == 'opencv':
        ret, frame = camera.read()
        if ret:
            return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

test_frame = get_frame()

if test_frame is not None:
    print(f"✓ Frame captured: {test_frame.shape}")
    
    # Display frame
    _, buffer = cv2.imencode('.jpg', cv2.cvtColor(test_frame, cv2.COLOR_RGB2BGR))
    display(Image(data=buffer.tobytes()))
else:
    print("ERROR: Could not capture frame")

## 4. Position Robot

**MANUAL STEP:**
1. Place the robot ~30 cm from the basket
2. Point the camera at the basket
3. Make sure the basket is visible in the frame above
4. Run the next cell when ready

In [ ]:
print("Ready to calibrate!")
print("Make sure:")
print("  - Robot is ~30 cm from basket")
print("  - Basket is centered in camera view")
print("  - Good lighting on basket")
print("\nPress the button below when ready...")

ready_button = widgets.Button(description="Start Calibration")
output = widgets.Output()

def on_ready_clicked(b):
    with output:
        print("Starting calibration...")

ready_button.on_click(on_ready_clicked)
display(ready_button, output)

## 5. Capture Calibration Frames

In [ ]:
print("Capturing 10 frames...")

calibration_frames = []

for i in range(10):
    frame = get_frame()
    if frame is not None:
        # Convert RGB to BGR for OpenCV processing
        bgr_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        calibration_frames.append(bgr_frame)
        print(f"  Frame {i+1}/10 captured")
    else:
        print(f"  Frame {i+1}/10 FAILED")
    
    time.sleep(0.2)  # 200ms between frames

print(f"\nCaptured {len(calibration_frames)} frames")

## 6. Run Calibration

In [ ]:
# Initialize detector
detector = BasketDetector()

# Run calibration
calibration_result = detector.calibrate(calibration_frames, verbose=True)

if calibration_result is None:
    print("\nCALIBRATION FAILED!")
    print("Possible issues:")
    print("  - Basket not visible in frames")
    print("  - Lighting too dark/bright")
    print("  - Wrong HSV range for gray color")
    print("\nTry adjusting robot position and re-run.")
else:
    print("\n✓ CALIBRATION SUCCESSFUL!")

## 7. Visualize Detection

In [ ]:
if calibration_result is not None:
    # Test detection on latest frame
    test_frame_bgr = calibration_frames[-1]
    detection = detector.detect(test_frame_bgr, use_calibration=True)
    
    # Draw overlay
    overlay = detector.draw_detection(test_frame_bgr, detection)
    
    # Display
    _, buffer = cv2.imencode('.jpg', overlay)
    display(Image(data=buffer.tobytes()))
    
    print("\nDetection result:")
    print(f"  Found: {detection['basket_found']}")
    if detection['basket_found']:
        print(f"  Centroid: {detection['centroid']}")
        print(f"  Bearing: {detection['bearing']:.1f}°")
        print(f"  Distance: {detection['distance']:.1f} cm")

## 8. Save to Config

In [ ]:
if calibration_result is not None:
    config_path = os.path.join(project_root, 'config.yaml')
    
    # Load existing config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Update basket section
    if 'basket' not in config:
        config['basket'] = {}
    
    config['basket']['size_px'] = float(calibration_result['size_px'])
    config['basket']['location_x'] = int(calibration_result['location_x'])
    config['basket']['location_y'] = int(calibration_result['location_y'])
    
    # Save config
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print("✓ Calibration saved to config.yaml")
    print(f"\nBasket parameters:")
    print(f"  size_px: {calibration_result['size_px']:.0f}")
    print(f"  location_x: {calibration_result['location_x']}")
    print(f"  location_y: {calibration_result['location_y']}")
else:
    print("Calibration failed, nothing to save")

## 9. Cleanup

In [ ]:
# Release camera
if camera_source == 'opencv' and camera is not None:
    camera.release()

print("Camera released. Calibration complete!")
print("\nNext step: Run 01_calibrate_colors.ipynb to tune ball detection")